In [20]:
import json
import os
from pathlib import Path
import grpc
import pandas as pd
import networkx as nx
from pyvis.network import Network
from senzing import SzEngine, SzError, SzEngineFlags
from senzing_grpc import SzAbstractFactoryGrpc
from IPython.display import display, HTML

# Get connection info from environment
SENZING_HOST = os.getenv('SENZING_GRPC_HOST', 'senzing')
SENZING_PORT = os.getenv('SENZING_GRPC_PORT', '8261')

print(f"Connecting to Senzing at {SENZING_HOST}:{SENZING_PORT}")

# Create gRPC channel and engine
grpc_url = f"{SENZING_HOST}:{SENZING_PORT}"
grpc_channel = grpc.insecure_channel(grpc_url)
sz_abstract_factory = SzAbstractFactoryGrpc(grpc_channel)
sz_engine = sz_abstract_factory.create_engine()

print("Connected to Senzing successfully")

Connecting to Senzing at senzing:8261
Connected to Senzing successfully


## Export All Resolved Entities from Senzing

Streams the full entity report out of Senzing using `export_json_entity_report()`, requesting the raw JSON for each source record alongside the resolved entity data.  The record-level JSON is important because it is where relationship data and name fields live.

In [10]:
print("Exporting entity data from Senzing with full details...")

entities = []
export_handle = sz_engine.export_json_entity_report(
    flags=SzEngineFlags.SZ_EXPORT_INCLUDE_ALL_ENTITIES |
          SzEngineFlags.SZ_ENTITY_INCLUDE_RECORD_JSON_DATA |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ALL_RELATIONS |
          SzEngineFlags.SZ_ENTITY_INCLUDE_ENTITY_NAME |
          SzEngineFlags.SZ_ENTITY_INCLUDE_RELATED_MATCHING_INFO
)
count = 0

while True:
    try:
        entity_json = sz_engine.fetch_next(export_handle)
        if not entity_json:
            break
        
        entity = json.loads(entity_json)
        entities.append(entity)
        count += 1
        
        if count % 100 == 0:
            print(f"  Exported {count} entities...", end='\r')
    except StopIteration:
        break

sz_engine.close_export_report(export_handle)
print(f"\nExported {len(entities)} entities total")

# Check how many have relationships
with_rels = sum(1 for e in entities if e.get('RELATED_ENTITIES'))
print(f"Entities with relationships: {with_rels}")

Exporting entity data from Senzing with full details...
  Exported 100 entities...
Exported 196 entities total
Entities with relationships: 189


## Dataset Overview

Pulls workload statistics from Senzing and compares total records loaded against total unique entities to show how much merging happened.  The gap between those two numbers is the whole point of entity resolution.

In [11]:
print("Dataset Overview:")
print("="*60)

stats = sz_engine.get_stats()
stats_dict = json.loads(stats)

if 'workload' in stats_dict:
    total_records = stats_dict['workload'].get('loadedRecords', 0)
    print(f"Total records in database: {total_records:,}")

# This should come after your export cell
num_entities = len(entities)
print(f"Total unique entities:     {num_entities:,}")
print(f"Records merged:            {total_records - num_entities:,}")
print("="*60)

Dataset Overview:
Total records in database: 282
Total unique entities:     196
Records merged:            86


## Inspect a Sample Entity's Structure

Prints the key fields of the first exported entity so you can see exactly what Senzing gives back: entity ID, name, record count, and any relationships.  This is a good reference before we start pulling these fields into graph nodes.

In [12]:
if entities:
    # Find an entity that has relationships for a better sample
    sample = None
    for e in entities:
        if e.get('RELATED_ENTITIES'):
            sample = e
            break
    if not sample:
        sample = entities[0]
    
    print(f"Entity ID: {sample.get('RESOLVED_ENTITY', {}).get('ENTITY_ID')}")
    print(f"Entity Name: {sample.get('RESOLVED_ENTITY', {}).get('ENTITY_NAME')}")
    
    # Check for records
    records = sample.get('RESOLVED_ENTITY', {}).get('RECORDS', [])
    print(f"Number of records: {len(records)}")
    
    # Check for related entities (from export flags)
    related = sample.get('RELATED_ENTITIES', [])
    print(f"Number of related entities: {len(related)}")
    
    print("\nFirst few records:")
    for rec in records[:3]:
        print(f"  {rec.get('DATA_SOURCE')}: {rec.get('RECORD_ID')}")
    
    if related:
        print("\nSample related entity:")
        print(json.dumps(related[0], indent=2))
else:
    print("No entities found")

Entity ID: 100001
Entity Name: Abassin Badshah
Number of records: 3
Number of related entities: 5

First few records:
  OPEN-SANCTIONS: NK-25vyVFzt8vdJGgAXMRTwTJ
  OPEN-OWNERSHIP: 17207853441353212969
  OPEN-OWNERSHIP: 6747548100436839873

Sample related entity:
{
  "ENTITY_ID": 100002,
  "MATCH_LEVEL_CODE": "DISCLOSED",
  "MATCH_KEY": "+ADDRESS+OPEN-SANCTIONS(DIRECTORSHIP:)-RECORD_TYPE",
  "ERRULE_CODE": "DISCLOSED",
  "IS_DISCLOSED": 1,
  "IS_AMBIGUOUS": 0
}


## Build the Entity-Level Graph with NetworkX

Creates a NetworkX graph where each node is a resolved entity and each edge is a relationship discovered by Senzing between two distinct entities. This is the clean, post-resolution view: multiple source records that describe the same real-world person or organization have already been collapsed into a single node. The node size reflects how many records were merged.

Edges here represent Senzing's "related" connections — entities that share some features (e.g., name, address) but not enough to be considered the same entity. These are business relationships like shareholding, directorship, or voting rights. Isolated nodes (with no edges) are entities that Senzing resolved from their source records but that share no features with any other entity in the dataset.

Note: this graph does **not** show the resolution process itself. To see which raw records were merged into each entity, see the True Combined Graph below.

In [16]:
G = nx.Graph()

# Build a lookup of record_id -> entity_id for name extraction
record_to_entity = {}
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    for rec in entity_data.get('RECORDS', []):
        record_to_entity[rec.get('RECORD_ID')] = entity_id

# Add nodes for each entity
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    entity_name = entity_data.get('ENTITY_NAME')
    
    # Get entity type and more details from records
    records = entity_data.get('RECORDS', [])
    record_type = 'UNKNOWN'
    data_sources = []
    addresses = []
    identifiers = []
    names = []
    
    if records:
        first_record = records[0]
        json_data = first_record.get('JSON_DATA', {})
        
        # v4 format: RECORD_TYPE is inside FEATURES array
        features = json_data.get('FEATURES', [])
        for feat in features:
            if 'RECORD_TYPE' in feat:
                record_type = feat['RECORD_TYPE']
                break
        
        # Extract names from FEATURES array (v4 format)
        for feat in features:
            if feat.get('NAME_FULL'):
                names.append(feat['NAME_FULL'])
            elif feat.get('NAME_ORG'):
                names.append(feat['NAME_ORG'])
            elif feat.get('PRIMARY_NAME_ORG'):
                names.append(feat['PRIMARY_NAME_ORG'])
        
        # Also check top-level and NAMES array as fallback
        if not names:
            if json_data.get('PRIMARY_NAME_FULL'):
                names.append(json_data['PRIMARY_NAME_FULL'])
            for name_obj in json_data.get('NAMES', []):
                full_name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
                if full_name:
                    names.append(full_name)
        
        # Collect info from all records
        for rec in records:
            data_sources.append(rec.get('DATA_SOURCE'))
            rec_json = rec.get('JSON_DATA', {})
            
            # Get addresses from FEATURES (v4) or ADDRESSES
            for feat in rec_json.get('FEATURES', []):
                if feat.get('ADDR_FULL') and len(addresses) < 1:
                    addresses.append(feat['ADDR_FULL'])
            for addr in rec_json.get('ADDRESSES', []):
                if addr.get('ADDR_FULL') and len(addresses) < 1:
                    addresses.append(addr['ADDR_FULL'])
    
    data_sources = list(set(data_sources))
    
    # Use ENTITY_NAME from export first, then fall back to FEATURES extraction
    primary_name = entity_name or (names[0] if names else None) or f"Entity {entity_id}"
    
    tooltip_parts = [
        primary_name,
        f"Type: {record_type}",
        f"Entity ID: {entity_id}",
        f"Records merged: {len(records)}",
        f"Data sources: {', '.join(data_sources)}"
    ]
    if len(names) > 1:
        tooltip_parts.append(f"Also known as: {', '.join(names[1:3])}")
    if addresses:
        tooltip_parts.append(f"Address: {addresses[0][:60]}")
    
    tooltip = "\n".join(tooltip_parts)
    
    display_label = primary_name
    if len(display_label) > 35:
        display_label = display_label[:32] + "..."
    
    G.add_node(
        entity_id,
        label=display_label,
        title=tooltip,
        type=record_type,
        data_sources=data_sources,
        num_records=len(records),
        full_name=primary_name
    )

print(f"Added {G.number_of_nodes()} nodes")

# Add edges from Senzing's RELATED_ENTITIES (resolved relationships)
edges_added = 0
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    
    for rel in entity.get('RELATED_ENTITIES', []):
        related_id = rel.get('ENTITY_ID')
        match_key = rel.get('MATCH_KEY', 'related')
        
        # Only add edge once (avoid duplicates since both sides report the relationship)
        if entity_id < related_id and G.has_node(related_id):
            G.add_edge(
                entity_id,
                related_id,
                relationship=match_key
            )
            edges_added += 1

print(f"Added {edges_added} edges")
print(f"\nGraph statistics:")
print(f"  Nodes: {G.number_of_nodes()}")
print(f"  Edges: {G.number_of_edges()}")
print(f"  Connected components: {nx.number_connected_components(G)}")

Added 196 nodes
Added 492 edges

Graph statistics:
  Nodes: 196
  Edges: 492
  Connected components: 25


## Visualize the Entity Graph with PyVis

Renders the entity-level graph as an interactive HTML visualization using Barnes-Hut physics for layout.  Orange nodes are people while blue nodes are organizations.  The node size scales with the number of records that were merged into that entity.  Hover over any node to see its details.

In [17]:
net = Network(
    height="1200px",  # Taller
    width="100%",
    bgcolor="#ffffff",
    font_color="#000000",
    notebook=True
)

# Set physics for better layout
net.barnes_hut(
    gravity=-5000,
    central_gravity=0.3,
    spring_length=100,
    spring_strength=0.001,
    damping=0.09
)

# Color nodes by type
color_map = {
    'PERSON': '#ff7f0e',
    'ORGANIZATION': '#1f77b4',
    'UNKNOWN': '#7f7f7f'
}

# Add nodes with styling
for node_id, node_data in G.nodes(data=True):
    node_type = node_data.get('type', 'UNKNOWN')
    num_records = node_data.get('num_records', 1)
    size = 10 + (num_records * 3)
    
    net.add_node(
        node_id,
        label=node_data.get('label', str(node_id)),
        title=node_data.get('title', ''),
        color=color_map.get(node_type, '#7f7f7f'),
        size=size
    )

# Add edges with labels
for source, target, edge_data in G.edges(data=True):
    relationship = edge_data.get('relationship', 'related')
    label = relationship.split()[0] if relationship else 'related'
    
    net.add_edge(
        source,
        target,
        label=label,
        title=relationship,
        color='#888888',
        font={'size': 10, 'color': '#333333'}
    )

# Save the graph
output_file = 'entity_graph.html'
net.save_graph(output_file)

print(f"\nVisualization saved successfully")
print("\nLegend:")
print("  Orange nodes = Persons")
print("  Blue nodes = Organizations")
print("  Node size = Number of records merged")
print("  Edge labels = Relationship type")
print(f"\nGraph contains {G.number_of_nodes()} nodes and {G.number_of_edges()} edges")

# Display with larger IFrame
from IPython.display import IFrame
display(IFrame(src=output_file, width='100%', height=1200))


Visualization saved successfully

Legend:
  Orange nodes = Persons
  Blue nodes = Organizations
  Node size = Number of records merged
  Edge labels = Relationship type

Graph contains 196 nodes and 492 edges


## Build the True Combined Graph (Entities + Records + Relationships)

Constructs a two-layer graph that makes both the results and the process of entity resolution visible:

- **Entity nodes** (large shapes) represent resolved entities — the same nodes as in the entity-level graph above. Their shape encodes how they were resolved: triangles are single-record entities (no merging), diamonds have multiple records merged from the same data source, and stars represent cross-source resolutions where records from both Open Ownership and Open Sanctions were merged into one entity.
- **Record nodes** (small dots) represent the original source records before resolution. Blue dots come from Open Ownership, red dots from Open Sanctions.
- **Gray dashed edges** connect each record node to the entity it was resolved into, making the merging process explicit. An entity with three gray lines converging on it means three separate records were identified as the same real-world person or organization.
- **Red solid edges** are the business relationships between entities (shareholding, directorship, etc.) — identical to the edges in the entity-level graph above.

This graph answers two questions at once: "who is related to whom?" (red edges) and "how did Senzing determine who is who?" (gray dashed edges linking records to entities).

In [23]:
G_true_combined = nx.Graph()

entity_info = {}
entities_added = 0
records_added = 0

for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    records = entity_data.get('RECORDS', [])
    
    if not records:
        continue
    
    first_record = records[0]
    json_data = first_record.get('JSON_DATA', {})
    
    # v4 format: extract type and name from FEATURES
    record_type = 'UNKNOWN'
    name = None
    features = json_data.get('FEATURES', [])
    for feat in features:
        if 'RECORD_TYPE' in feat:
            record_type = feat['RECORD_TYPE']
        if not name:
            name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
    
    # Fallback to top-level fields
    if not name:
        name = json_data.get('PRIMARY_NAME_FULL')
    if not name:
        for name_obj in json_data.get('NAMES', []):
            name = name_obj.get('NAME_FULL') or name_obj.get('NAME_ORG') or name_obj.get('PRIMARY_NAME_ORG')
            if name:
                break
    if not name:
        name = f"Entity {entity_id}"
    
    data_sources = list(set([r.get('DATA_SOURCE') for r in records]))
    is_cross_source = len(data_sources) > 1
    
    entity_info[entity_id] = {
        'name': name,
        'type': record_type,
        'num_records': len(records),
        'is_cross_source': is_cross_source,
        'data_sources': data_sources
    }
    
    tooltip_parts = [
        name,
        f"Type: {record_type}",
        f"Entity ID: {entity_id}",
        f"Records merged: {len(records)}",
        f"Data sources: {', '.join(data_sources)}"
    ]
    if is_cross_source:
        tooltip_parts.append("⚠️ CROSS-SOURCE RESOLUTION")
    tooltip = "\n".join(tooltip_parts)
    
    display_label = name[:30] + "..." if len(name) > 30 else name
    
    entity_node_id = f"entity_{entity_id}"
    G_true_combined.add_node(
        entity_node_id,
        label=display_label,
        title=tooltip,
        node_type='entity',
        type=record_type,
        num_records=len(records),
        is_cross_source=is_cross_source,
        entity_id=entity_id
    )
    entities_added += 1
    
    # Add record nodes and connect to entity
    for rec in records:
        rec_id = rec.get('RECORD_ID')
        rec_source = rec.get('DATA_SOURCE')
        rec_json = rec.get('JSON_DATA', {})
        
        # Get record name from FEATURES (v4)
        rec_name = None
        rec_type = 'UNKNOWN'
        for feat in rec_json.get('FEATURES', []):
            if 'RECORD_TYPE' in feat:
                rec_type = feat['RECORD_TYPE']
            if not rec_name:
                rec_name = feat.get('NAME_FULL') or feat.get('NAME_ORG') or feat.get('PRIMARY_NAME_ORG')
        if not rec_name:
            rec_name = name
        
        rec_tooltip = "\n".join([
            f"Record: {rec_name}",
            f"Source: {rec_source}",
            f"Type: {rec_type}",
            f"Record ID: {rec_id}",
            f"Resolves to: {name}"
        ])
        
        rec_label = rec_name[:20] + "..." if len(rec_name) > 20 else rec_name
        
        record_node_id = f"record_{rec_source}_{rec_id}"
        G_true_combined.add_node(
            record_node_id,
            label=rec_label,
            title=rec_tooltip,
            node_type='record',
            data_source=rec_source,
            type=rec_type
        )
        records_added += 1
        
        G_true_combined.add_edge(
            record_node_id,
            entity_node_id,
            edge_type='resolution'
        )

print(f"Added {entities_added} entity nodes")
print(f"Added {records_added} record nodes")
print(f"Added {G_true_combined.number_of_edges()} resolution edges")

# Add relationship edges from Senzing's RELATED_ENTITIES
relationship_edges = 0
for entity in entities:
    entity_data = entity.get('RESOLVED_ENTITY', {})
    entity_id = entity_data.get('ENTITY_ID')
    
    for rel in entity.get('RELATED_ENTITIES', []):
        related_id = rel.get('ENTITY_ID')
        match_key = rel.get('MATCH_KEY', 'related')
        
        if entity_id < related_id:
            anchor_node = f"entity_{entity_id}"
            target_node = f"entity_{related_id}"
            if G_true_combined.has_node(target_node):
                G_true_combined.add_edge(
                    anchor_node,
                    target_node,
                    edge_type='relationship',
                    relationship=match_key
                )
                relationship_edges += 1

print(f"Added {relationship_edges} relationship edges")
print(f"\nTrue Combined Graph Statistics:")
print(f"  Total nodes: {G_true_combined.number_of_nodes()}")
print(f"  Total edges: {G_true_combined.number_of_edges()}")
print(f"  Entity nodes: {entities_added}")
print(f"  Record nodes: {records_added}")
print(f"  Resolution edges: {G_true_combined.number_of_edges() - relationship_edges}")
print(f"  Relationship edges: {relationship_edges}")

Added 196 entity nodes
Added 282 record nodes
Added 282 resolution edges
Added 492 relationship edges

True Combined Graph Statistics:
  Total nodes: 478
  Total edges: 774
  Entity nodes: 196
  Record nodes: 282
  Resolution edges: 282
  Relationship edges: 492


## Visualize the True Combined Graph

Renders the full two-layer graph with PyVis and injects a small JavaScript snippet to automatically disable physics after 10 seconds so the layout settles and nodes stay where you drag them.  Stars with red borders are the most interesting nodes: they represent entities that Senzing matched across both the Open Ownership and Open Sanctions datasets.

In [32]:
net_true = Network(
    height="1200px",
    width="100%",
    bgcolor="#ffffff",
    font_color="#000000",
    notebook=True
)

# Physics tuned to spread clusters apart and reduce edge overlap
net_true.repulsion(
    node_distance=250,
    central_gravity=0.1,
    spring_length=300,
    spring_strength=0.02,
    damping=0.3
)

# Entity colors by type
entity_color_map = {
    'PERSON': '#ff7f0e',      # Orange
    'ORGANIZATION': '#1f77b4', # Blue
    'UNKNOWN': '#7f7f7f'       # Gray
}

# Record colors by data source (green/red to avoid colliding with entity colors)
record_color_map = {
    'OPEN-OWNERSHIP': '#2ca02c',  # Green
    'OPEN-SANCTIONS': '#e74c3c'   # Red
}

# Add nodes
for node_id, node_data in G_true_combined.nodes(data=True):
    if node_data.get('node_type') == 'entity':
        # Entity nodes
        node_type = node_data.get('type', 'UNKNOWN')
        num_records = node_data.get('num_records', 1)
        is_cross_source = node_data.get('is_cross_source', False)
        
        size = 30 + (num_records * 5)
        
        if is_cross_source:
            shape = 'star'
        elif num_records > 1:
            shape = 'diamond'
        else:
            shape = 'triangle'
        
        color = entity_color_map.get(node_type, '#7f7f7f')
        if is_cross_source:
            border_color = '#e74c3c'
            border_width = 5
        elif num_records > 1:
            border_color = '#2ecc71'
            border_width = 3
        else:
            border_color = color
            border_width = 1
        
        net_true.add_node(
            node_id,
            label=node_data.get('label', ''),
            title=node_data.get('title', ''),
            color={'background': color, 'border': border_color},
            size=size,
            shape=shape,
            borderWidth=border_width
        )
    else:
        # Record nodes
        data_source = node_data.get('data_source', 'UNKNOWN')
        color = record_color_map.get(data_source, '#95a5a6')
        
        net_true.add_node(
            node_id,
            label=node_data.get('label', ''),
            title=node_data.get('title', ''),
            color=color,
            size=15,
            shape='dot'
        )

# Add edges
for source, target, edge_data in G_true_combined.edges(data=True):
    edge_type = edge_data.get('edge_type', 'resolution')
    
    if edge_type == 'relationship':
        relationship = edge_data.get('relationship', 'related')
        label = relationship.split()[0] if relationship else 'related'
        
        net_true.add_edge(
            source,
            target,
            label=label,
            title=relationship,
            color='#e74c3c',
            width=3,
            font={'size': 10, 'color': '#e74c3c'},
            smooth={'type': 'curvedCW', 'roundness': 0.1}
        )
    else:
        net_true.add_edge(
            source,
            target,
            title='resolved to',
            color='#cccccc',
            width=1,
            dashes=True,
            smooth=False
        )

# Save and modify to auto-disable physics
output_file = 'true_combined_graph.html'
net_true.save_graph(output_file)

# Add code to disable physics after 15 seconds (longer to let repulsion settle)
with open(output_file, 'r') as f:
    html_content = f.read()

physics_auto_off = """
<script type="text/javascript">
  // Disable physics after 15 seconds
  setTimeout(function() {
    network.setOptions({ physics: false });
    console.log("Physics auto-disabled after 15 seconds");
  }, 15000);
</script>
"""

html_content = html_content.replace('</body>', physics_auto_off + '</body>')

with open(output_file, 'w') as f:
    f.write(html_content)

print(f"\nVisualization saved successfully")
print("\n" + "="*70)
print("HOW TO READ THIS GRAPH")
print("="*70)
print("\nLARGE SHAPES = RESOLVED ENTITIES (after entity resolution)")
print("  Triangles:")
print("    - Single record, no merging happened")
print("    - Orange = Person, Blue = Organization")
print("  ")
print("  Diamonds (GREEN BORDER):")
print("    - Multiple records from SAME data source merged together")
print("    - Example: 3 OPEN-OWNERSHIP records identified as same company")
print("    - Orange = Person, Blue = Organization")
print("  ")
print("  Stars (RED BORDER):")
print("    - Records from DIFFERENT data sources merged together")
print("    - Example: Person in OPEN-SANCTIONS matched to director in OPEN-OWNERSHIP")
print("    - Orange = Person, Blue = Organization")
print("    - These are the most interesting - cross-dataset connections!")
print("\nSMALL DOTS = ORIGINAL RECORDS (before entity resolution)")
print("  Green dots = Records from OPEN-OWNERSHIP dataset")
print("  Red dots = Records from OPEN-SANCTIONS dataset")
print("\nLINES = CONNECTIONS")
print("  Gray dashed lines:")
print("    - Connect original records to their resolved entity")
print("    - Show which records Senzing merged together")
print("  ")
print("  Red solid lines (with labels):")
print("    - Show business relationships between entities")
print("    - Labels show relationship type: shareholding, Directorship, voting_rights, etc.")
print("\nINTERACTION:")
print("  - Physics will settle in about 15 seconds")
print("  - Then auto-disables so nodes stay where you drag them")
print("  - Scroll to zoom, drag background to pan")
print("\nEXAMPLE: A star with red border surrounded by green and red dots")
print("  = Person/org found in BOTH datasets, with multiple records merged")
print("  = The dots show the original records that were combined")
print("  = Red lines show their business connections to other entities")
print("="*70)

# Display
from IPython.display import IFrame
display(IFrame(src=output_file, width='100%', height=1200))


Visualization saved successfully

HOW TO READ THIS GRAPH

LARGE SHAPES = RESOLVED ENTITIES (after entity resolution)
  Triangles:
    - Single record, no merging happened
    - Orange = Person, Blue = Organization
  
  Diamonds (GREEN BORDER):
    - Multiple records from SAME data source merged together
    - Example: 3 OPEN-OWNERSHIP records identified as same company
    - Orange = Person, Blue = Organization
  
  Stars (RED BORDER):
    - Records from DIFFERENT data sources merged together
    - Example: Person in OPEN-SANCTIONS matched to director in OPEN-OWNERSHIP
    - Orange = Person, Blue = Organization
    - These are the most interesting - cross-dataset connections!

SMALL DOTS = ORIGINAL RECORDS (before entity resolution)
  Green dots = Records from OPEN-OWNERSHIP dataset
  Red dots = Records from OPEN-SANCTIONS dataset

LINES = CONNECTIONS
  Gray dashed lines:
    - Connect original records to their resolved entity
    - Show which records Senzing merged together
  
  Red 

## Explore a Subgraph by Entity Name

Extracts the connected component containing a given entity from the True Combined Graph and visualizes it in isolation. Enter the name of any person or organization (case-insensitive substring match) to see just that entity's neighborhood — all entities reachable from it via relationships, plus their source records. This makes it much easier to trace a single entity's connections without the clutter of the full graph.

In [38]:
# --- Change this name to explore a different entity's subgraph ---
SEARCH_NAME = "Said Kerimov"

# Find matching nodes by case-insensitive substring match across both entity and record nodes
search_lower = SEARCH_NAME.lower()
matching_entity_nodes = set()

for node_id, node_data in G_true_combined.nodes(data=True):
    label = node_data.get('label', '').lower()
    title = node_data.get('title', '').lower()
    if search_lower in label or search_lower in title:
        if node_data.get('node_type') == 'entity':
            matching_entity_nodes.add(node_id)
        elif node_data.get('node_type') == 'record':
            # Trace the record back to its parent entity
            for neighbor in G_true_combined.neighbors(node_id):
                neighbor_data = G_true_combined.nodes[neighbor]
                if neighbor_data.get('node_type') == 'entity':
                    matching_entity_nodes.add(neighbor)

matching_nodes = list(matching_entity_nodes)

if not matching_nodes:
    print(f"No entity found matching '{SEARCH_NAME}'.")
    print("\nAvailable entity names:")
    entity_names = sorted([
        d.get('label', '') for _, d in G_true_combined.nodes(data=True)
        if d.get('node_type') == 'entity'
    ])
    for name in entity_names:
        print(f"  {name}")
else:
    # Get all nodes reachable from matching nodes (connected component)
    reachable = set()
    for start_node in matching_nodes:
        reachable.update(nx.node_connected_component(G_true_combined, start_node))
    
    # Extract the subgraph
    G_sub = G_true_combined.subgraph(reachable).copy()
    
    print(f"Found {len(matching_nodes)} entity node(s) matching '{SEARCH_NAME}':")
    for mn in matching_nodes:
        print(f"  {G_true_combined.nodes[mn].get('label', mn)}")
    
    entity_count = sum(1 for _, d in G_sub.nodes(data=True) if d.get('node_type') == 'entity')
    record_count = sum(1 for _, d in G_sub.nodes(data=True) if d.get('node_type') == 'record')
    rel_edges = sum(1 for _, _, d in G_sub.edges(data=True) if d.get('edge_type') == 'relationship')
    res_edges = sum(1 for _, _, d in G_sub.edges(data=True) if d.get('edge_type') == 'resolution')
    
    print(f"\nSubgraph contains:")
    print(f"  {entity_count} entities, {record_count} records")
    print(f"  {rel_edges} relationship edges, {res_edges} resolution edges")
    
    # Visualize with the same styling as the full True Combined Graph
    net_sub = Network(
        height="1200px",
        width="100%",
        bgcolor="#ffffff",
        font_color="#000000",
        notebook=True
    )
    
    net_sub.repulsion(
        node_distance=250,
        central_gravity=0.1,
        spring_length=300,
        spring_strength=0.02,
        damping=0.3
    )
    
    entity_color_map = {
        'PERSON': '#ff7f0e',
        'ORGANIZATION': '#1f77b4',
        'UNKNOWN': '#7f7f7f'
    }
    record_color_map = {
        'OPEN-OWNERSHIP': '#2ca02c',
        'OPEN-SANCTIONS': '#e74c3c'
    }
    
    for node_id, node_data in G_sub.nodes(data=True):
        if node_data.get('node_type') == 'entity':
            node_type = node_data.get('type', 'UNKNOWN')
            num_records = node_data.get('num_records', 1)
            is_cross_source = node_data.get('is_cross_source', False)
            
            size = 30 + (num_records * 5)
            
            if is_cross_source:
                shape = 'star'
            elif num_records > 1:
                shape = 'diamond'
            else:
                shape = 'triangle'
            
            color = entity_color_map.get(node_type, '#7f7f7f')
            if is_cross_source:
                border_color = '#e74c3c'
                border_width = 5
            elif num_records > 1:
                border_color = '#2ecc71'
                border_width = 3
            else:
                border_color = color
                border_width = 1
            
            # Highlight the searched entity by making it larger
            if node_id in matching_nodes:
                size = size + 20
            
            net_sub.add_node(
                node_id,
                label=node_data.get('label', ''),
                title=node_data.get('title', ''),
                color={'background': color, 'border': border_color},
                size=size,
                shape=shape,
                borderWidth=border_width
            )
        else:
            data_source = node_data.get('data_source', 'UNKNOWN')
            color = record_color_map.get(data_source, '#95a5a6')
            
            net_sub.add_node(
                node_id,
                label=node_data.get('label', ''),
                title=node_data.get('title', ''),
                color=color,
                size=15,
                shape='dot'
            )
    
    for source, target, edge_data in G_sub.edges(data=True):
        edge_type = edge_data.get('edge_type', 'resolution')
        
        if edge_type == 'relationship':
            relationship = edge_data.get('relationship', 'related')
            label = relationship.split()[0] if relationship else 'related'
            
            net_sub.add_edge(
                source,
                target,
                label=label,
                title=relationship,
                color='#e74c3c',
                width=3,
                font={'size': 10, 'color': '#e74c3c'},
                smooth={'type': 'curvedCW', 'roundness': 0.1}
            )
        else:
            net_sub.add_edge(
                source,
                target,
                title='resolved to',
                color='#cccccc',
                width=1,
                dashes=True,
                smooth=False
            )
    
    output_file = 'subgraph.html'
    net_sub.save_graph(output_file)
    
    with open(output_file, 'r') as f:
        html_content = f.read()
    
    physics_auto_off = """
    <script type="text/javascript">
      setTimeout(function() {
        network.setOptions({ physics: false });
        console.log("Physics auto-disabled after 15 seconds");
      }, 15000);
    </script>
    """
    html_content = html_content.replace('</body>', physics_auto_off + '</body>')
    
    with open(output_file, 'w') as f:
        f.write(html_content)
    
    print(f"\nSubgraph visualization saved. Searched entity is larger than the others.")
    
    from IPython.display import IFrame
    display(IFrame(src=output_file, width='100%', height=1200))

Found 2 entity node(s) matching 'Said Kerimov':
  Said Kerimov
  Said Kerimov

Subgraph contains:
  37 entities, 49 records
  121 relationship edges, 49 resolution edges

Subgraph visualization saved. Searched entity is larger than the others.
